## Расчет метрик (только для начального, переименованного) датасета

In [1]:
import pandas as pd
import re

*Для расчета метрики используем весь представленный датасет 1200 фото. Конечно, для obb модели это не корректно, так как она обучалась на первых 200 фото, но ocr их не видела так как обучалась на синтетических датасетах, а очень интересно как она справится на всем объеме*

In [2]:
# Загрузим сформированный пайплайном отчет*
df_metrics = pd.read_csv("result_for_metrics_full_1200.csv")

In [3]:
df_metrics.columns

Index(['id', 'old_filename', 'result', 'confidence'], dtype='object')

In [4]:
df_metrics.head()

,id,old_filename,result,confidence
0,full_1200_1,MDZ086.JPG,0086,0.94
1,full_1200_2,MDZ086_1.JPG,0086,0.92
2,full_1200_3,MDZ086_2.JPG,0086,0.93
3,full_1200_4,MDZ087 _1.JPG,0087,0.96
4,full_1200_5,MDZ087 _10.JPG,0087,0.96


### Метрики для всей модели целиком (obb+ocr)

*Учитывает все кадры, в том числе кадры с реально отсутствующей табличкой, которые все равно считаем как ошибку модели*

*Для клиента не имеют значения промежуточные метрики самих моделей. Ему важно понимать сколько раз прогноз оказался верным. Поэтому для оценки возьмем Accuracy. Для ее расчета вынем из имени файла правильное значение и сравним с распознаным*

In [5]:
def get_original_number(filename):
    # Очистка от путей и расширений
    filename_str = str(filename)
    name_only = filename_str.split('/')[-1].split('\\')[-1].rsplit('.', 1)[0]
    base = name_only.split('_')[0]
    
    match = re.search(r'(\d+)([a-zA-Z]?)', base)
    if not match: return "unknown"
    
    digits, letter = match.group(1), match.group(2)
    combined = (digits + letter)
    
    if len(combined) >= 4:
        res = combined[:4]
    else:
        res = combined.zfill(4)
    return res.lower()

# 2. Создаем столбцы (ИСПРАВЛЕНО: добавлен .str перед .lower())
df_metrics['old_filename_number'] = df_metrics['old_filename'].apply(get_original_number)
df_metrics['result_clean'] = df_metrics['result'].astype(str).str.strip().str.lower()

# 3. Считаем совпадение (coincidence)
# 1 - если совпало идеально, 0 - если есть расхождение или "!"
df_metrics['coincidence'] = ((df_metrics['old_filename_number'] == df_metrics['result_clean']) & 
                             (~df_metrics['result_clean'].str.contains('!', na=False))).astype(int)

# 4. Итоговые метрики
total = len(df_metrics)
matches = df_metrics['coincidence'].sum()
accuracy = matches / total if total > 0 else 0


#print(f"--- АНАЛИЗ ПАРТИИ: {PREFIX} ---")
print(f"Всего строк: {total}")
print(f"Точность (Accuracy): {accuracy:.2%}")

Всего строк: 1200
Точность (Accuracy): 88.83%


In [6]:
# Вывод результата
df_metrics[20:30]

,id,old_filename,result,confidence,old_filename_number,result_clean,coincidence
20,full_1200_21,MDZ088 _3.JPG,0008,0.23,0088,0008,0
21,full_1200_22,MDZ088 _4.JPG,!MDZ088 _4.JPG,0.00,0088,!mdz088 _4.jpg,0
22,full_1200_23,MDZ088 _5.JPG,0088,0.96,0088,0088,1
23,full_1200_24,MDZ088 _6.JPG,!MDZ088 _6.JPG,0.41,0088,!mdz088 _6.jpg,0
24,full_1200_25,MDZ088 _7.JPG,1212,0.49,0088,1212,0
25,full_1200_26,MDZ088.JPG,0088,0.94,0088,0088,1
26,full_1200_27,MDZ088_1.JPG,0088,0.95,0088,0088,1
27,full_1200_28,MDZ089 _1.JPG,0089,0.95,0089,0089,1
28,full_1200_29,MDZ089 _2.JPG,0089,0.96,0089,0089,1
29,full_1200_30,MDZ089 _3.JPG,0089,0.94,0089,0089,1


*Вывод: На мой взгляд для реальной оценки работы модели с точки зрения клиента лучше метрики, чем Accuracy не придумать. По ней он может понять объем правильных ответов с учетом ошибок модели и некачественных данных*

*Вывод: Не вижу смысла считать F1-score и ROC-AUC для первой модели, которая ищет таблички. В нашем случае порог все равно нужно сдвигать в самый минимум, чтобы она находила все, что сможет. Если она найдет табличку там, где ее нет, то ocr все равно не пропустит и пометит как ошибку*

### Анализ только OCR (где кроп получен)

*Не учитываются случаи, где таблички не было вообще или она не была найдена моделью obb.
Хотя это не совсем чисто потому, что obb иногда дает не правильную рамку и у OCR нет шансов, но это компенсируется случаями, когда ocr на нормальном кропе не смог ничего прочитать. И тех и других случаев реально не много, поэтому метрику можно считать приближенной к реальности. А оценить отдель блок ocr имеет смысл*

*И, конечно, метрику по OCR также портят "бракованные" кадры. Где реально в кадр не попадает пол таблички или реальная цифра закрыта веткой на 60-70%*

In [7]:
# 1. Делаем срез: убираем строки, где OCR выдал "!" (наш маркер мусора/неудачи)
df_ocr_only = df_metrics[~df_metrics['result'].astype(str).str.contains('!', na=False)].copy()

if not df_ocr_only.empty:
    # 2. Считаем метрики для этого среза
    total_ocr = len(df_ocr_only)
    # Совпадение уже посчитано в coincidence, просто берем сумму по срезу
    matches_ocr = df_ocr_only['coincidence'].sum()
    
    accuracy_ocr = matches_ocr / total_ocr if total_ocr > 0 else 0
    
    print(f"=== АНАЛИЗ ЧИСТОГО OCR (строк без '!') ===")
    print(f"Количество строк в анализе: {total_ocr} (из {len(df_metrics)} исходных)")
    print(f"Accuracy (Чистое распознавание): {accuracy_ocr:.2%}")
        
    # Посмотрим на самые коварные ошибки (где OCR уверен, но не прав)
    print("\nТоп-5 'уверенных' ошибок OCR (высокий confidence при несовпадении):")
    tricky_errors = df_ocr_only[df_ocr_only['coincidence'] == 0].sort_values('confidence', ascending=False)
    display(tricky_errors.head(5))
else:
    print("Строк без '!' не найдено.")

=== АНАЛИЗ ЧИСТОГО OCR (строк без '!') ===
Количество строк в анализе: 1130 (из 1200 исходных)
Accuracy (Чистое распознавание): 94.34%

Топ-5 'уверенных' ошибок OCR (высокий confidence при несовпадении):


,id,old_filename,result,confidence,old_filename_number,result_clean,coincidence
1060,full_1200_1061,TUL843_3.JPG,0084,0.96,0843,0084,0
1013,full_1200_1014,TUL821_2.JPG,0082,0.95,0821,0082,0
207,full_1200_208,MDZ116_4.JPG,0016,0.94,0116,0016,0
260,full_1200_261,MDZ126_5.JPG,0016,0.93,0126,0016,0
578,full_1200_579,TUL004_2.JPG,0005,0.93,0004,0005,0


*Вывод: Несмотря на неплохую точность блока ocr, мы видим, что модель все-таки ошибается при высоком confidence. Так как в нашем случае важно именно не допустить ошибки, то не представляется возможным ориентироваться на confidence для принятия решения о переименовании файла*

*Предложение: добавить в приложение ручное подтверждение*